In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os, json
import numpy as np
import torch

In [ ]:
def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    return rows

absa_train_path = "/content/drive/MyDrive/FYP/absa/absa_train_big.jsonl"
absa_dev_path = "/content/drive/MyDrive/FYP/absa/absa_dev.jsonl"
absa_test_path = "/content/drive/MyDrive/FYP/absa/absa_test.jsonl"

train_rows = load_jsonl(absa_train_path)
dev_rows = load_jsonl(absa_dev_path)
test_rows = load_jsonl(absa_test_path)

In [ ]:
LABELS = ["negative", "neutral", "positive", "conflict"]
label2id = {lab:i for i, lab in enumerate(LABELS)}
id2label = {i:lab for lab, i in label2id.items()}

In [ ]:
def norm_pol(p):
    return p.strip().lower()

In [ ]:
def expand_absa(rows):
    out_text = []
    out_cat  = []
    out_lab  = []
    for r in rows:
        text = r["text"]
        for lab in r["labels"]:
            cat = lab["category"]
            pol = norm_pol(lab["polarity"])
            if pol not in label2id:
                continue
            out_text.append(text)
            out_cat.append(cat)
            out_lab.append(label2id[pol])
    return {"text": out_text, "category": out_cat, "label": out_lab}

In [ ]:
from datasets import Dataset
train_ds = Dataset.from_dict(expand_absa(train_rows))
dev_ds = Dataset.from_dict(expand_absa(dev_rows))
test_ds = Dataset.from_dict(expand_absa(test_rows))

In [ ]:
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)

In [ ]:
# model_name = "bert-base-multilingual-cased"
# model_name = "asafaya/bert-base-arabic"
model_name = "aubmindlab/bert-base-arabertv2"

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

config.json:   0%|          | 0.00/384 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/611 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [ ]:
def tokenize_fn(batch):
    return tokenizer(
        batch["category"],
        batch["text"],
        truncation=True,
        padding=False,
        max_length=256,
    )

In [ ]:
train_ds = train_ds.map(tokenize_fn, batched=True)
dev_ds   = dev_ds.map(tokenize_fn, batched=True)
test_ds  = test_ds.map(tokenize_fn, batched=True)

Map:   0%|          | 0/7635 [00:00<?, ? examples/s]

Map:   0%|          | 0/1122 [00:00<?, ? examples/s]

Map:   0%|          | 0/2158 [00:00<?, ? examples/s]

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(LABELS),
    id2label=id2label,
    label2id=label2id,
)

model.safetensors:   0%|          | 0.00/543M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: aubmindlab/bert-base-arabertv2
Key                                        | Status     | 
-------------------------------------------+------------+-
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider

In [ ]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
from sklearn.metrics import f1_score, accuracy_score, classification_report
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    macro = f1_score(labels, preds, average="macro")
    return {"acc": acc, "macro_f1": macro}

In [ ]:
out_dir = "/content/drive/MyDrive/FYP/absa/bert_arabert_conditioned_absa"

training_args = TrainingArguments(
    output_dir=out_dir,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    fp16=torch.cuda.is_available(),
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=dev_ds,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Acc,Macro F1
1,0.498572,0.575805,0.804813,0.397899
2,0.435252,0.564434,0.811052,0.396020
3,0.375448,0.556095,0.811943,0.417777


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=1434, training_loss=0.455821450285333, metrics={'train_runtime': 103.9715, 'train_samples_per_second': 220.301, 'train_steps_per_second': 13.792, 'total_flos': 1861580117039760.0, 'train_loss': 0.455821450285333, 'epoch': 3.0})

In [ ]:
test_metrics = trainer.evaluate(eval_dataset=test_ds)

In [ ]:
trainer.save_model(out_dir)
tokenizer.save_pretrained(out_dir)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/drive/MyDrive/FYP/absa/bert_arabert_conditioned_absa/tokenizer_config.json',
 '/content/drive/MyDrive/FYP/absa/bert_arabert_conditioned_absa/tokenizer.json')

In [ ]:
result = {
    "acc": test_metrics["eval_acc"],
    "macro_f1": test_metrics["eval_macro_f1"],
    "n": len(test_ds),
}

In [ ]:
save_path = "/content/drive/MyDrive/FYP/absa/bert_results/arabert_conditioned_absa_results.json"
os.makedirs(os.path.dirname(save_path), exist_ok=True)
with open(save_path, "w", encoding="utf-8") as f:
    json.dump(result, f, ensure_ascii=False, indent=4)